In [ ]:
import json
import logging
import warnings
from pathlib import Path

import joblib
from tqdm.auto import tqdm
import numpy as np

# Compatibility fix: IOHMM uses np.Infinity,
# which was removed in NumPy 2.0.
np.Infinity = np.inf

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.special import logsumexp
from statsmodels.genmod.families import Binomial

from IOHMM import UnSupervisedIOHMM, GLM, CrossEntropyMNL
from IOHMM.forward_backward import forward

from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    matthews_corrcoef,
    precision_recall_curve,
    roc_curve,
    fbeta_score,
    auc,
)

from sklearn.feature_selection import (
    SelectFromModel,
    VarianceThreshold,
)

from sklearn.model_selection import (
    ParameterGrid,
    GridSearchCV,
    RandomizedSearchCV,
    GroupKFold,
    GroupShuffleSplit,
)

from sklearn.calibration import calibration_curve
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.utils.class_weight import compute_class_weight

import optuna

from ingestion.load_openesm import load_curated_dataset

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

sns.set_style("whitegrid")

In [ ]:
# Reduce noisy third-party warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module=r"sklearn\.linear_model"
)

warnings.filterwarnings(
    "ignore",
    message="divide by zero encountered in log",
    category=RuntimeWarning,
    module=r"IOHMM\.IOHMM"
)

warnings.filterwarnings(
    "ignore",
    message="divide by zero encountered in scalar divide",
    category=RuntimeWarning,
    module=r"IOHMM\.linear_models"
)

In [2]:
df = load_curated_dataset("0006")

In [3]:
df.head()

,id,day,beep,responded,day_index,beep_index,prompt_index,day_squared,beep_squared,day_beep_interaction,days_since_first_observed,first_observed_day,last_observed_day,prompts_in_day,first_beep_of_day,last_beep_of_day,beep_position,days_since_previous_prompt,beep_difference_from_previous,beep_gap_from_previous_same_day,scheduled_beeps_in_day,scheduled_beep_position,scheduled_beep_position_rate,cumulative_responses,cumulative_prompts,cumulative_response_rate,prior_cumulative_responses,prior_cumulative_prompts,prior_cumulative_response_rate,prior_cumulative_nonresponses,prior_missed_beep_count,prior_smoothed_response_rate,prior_response_history_count,prev_1_response,prev_2_response,prev_3_response,prev_4_response,prev_5_response,prev_6_response,prev_response,prev_response_same_day,prior_3_prompt_response_rate,prior_6_prompt_response_rate,prior_12_prompt_response_rate,prior_ewm_3_prompt_response_rate,prior_ewm_6_prompt_response_rate,prior_response_streak,prior_nonresponse_streak,days_since_previous_response,prior_same_day_responses,prior_same_day_prompts,prior_same_day_nonresponses,prior_same_day_response_rate,prior_same_day_smoothed_response_rate,prior_same_day_response_streak,prior_same_day_nonresponse_streak,prior_same_day_2_prompt_response_rate,prior_same_day_3_prompt_response_rate,prior_same_day_ewm_3_response_rate,prior_same_day_ewm_6_response_rate,prior_same_beep_responses,prior_same_beep_observations,prior_same_beep_nonresponses,prior_same_beep_response_rate,prior_same_beep_smoothed_rate,prior_same_beep_response_streak,prior_same_beep_nonresponse_streak,prev_same_beep_response,days_since_previous_same_beep,prior_same_beep_ewm_3_response_rate,prior_same_beep_ewm_6_response_rate,prev_day_same_beep_response,prev_day_same_beep_available,responded_any_today,daily_response_rate,prior_day_response_rate,prior_days_with_response,prior_days_observed,prior_day_participation_rate,prev_day_responded,prev_day_response_rate,days_since_previous_observed_day,prior_consecutive_active_days,prior_consecutive_inactive_days,prior_response_rate_change,prior_3_day_response_rate,prior_3_day_participation_rate,prior_7_day_response_rate,prior_7_day_participation_rate,prior_ewm_3_day_response_rate,prior_ewm_6_day_response_rate,missed_day
0,1,1,1,1,1,1,1,1,1,1,0,1,0,6,1,0,0.166667,<NA>,<NA>,<NA>,6,1,0.166667,1,1,100.000000,0,0,NaN,0,0,50.000000,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,<NA>,0,0,0,NaN,50.000000,0,0,NaN,NaN,NaN,NaN,0,0,0,NaN,50.0,0,0,NaN,<NA>,NaN,NaN,NaN,0,1,66.666667,NaN,0.0,0,NaN,NaN,NaN,<NA>,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1,1,2,0,1,2,2,1,4,2,0,1,0,6,0,0,0.333333,0,1,1,6,2,0.333333,1,2,50.000000,1,1,100.000000,0,0,66.666667,1,1.0,NaN,NaN,NaN,NaN,NaN,1.0,1.0,100.000000,100.000000,100.000000,100.0,100.000000,1,0,0,1,1,0,100.000000,66.666667,1,0,100.0,100.000000,100.0,100.000000,0,0,0,NaN,50.0,0,0,NaN,<NA>,NaN,NaN,NaN,0,1,66.666667,NaN,0.0,0,NaN,NaN,NaN,<NA>,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,1,1,3,1,1,3,3,1,9,3,0,1,0,6,0,0,0.5,0,1,1,6,3,0.5,2,3,66.666667,1,2,50.000000,1,1,50.000000,2,0.0,1.0,NaN,NaN,NaN,NaN,0.0,0.0,50.000000,50.000000,50.000000,50.0,71.428571,0,1,0,1,2,1,50.000000,50.000000,0,1,50.0,50.000000,50.0,71.428571,0,0,0,NaN,50.0,0,0,NaN,<NA>,NaN,NaN,NaN,0,1,66.666667,NaN,0.0,0,NaN,NaN,NaN,<NA>,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,1,1,4,1,1,4,4,1,16,4,0,1,0,6,0,0,0.666667,0,1,1,6,4,0.666667,3,4,75.000000,2,3,66.666667,1,1,60.000000,3,1.0,0.0,1.0,NaN,NaN,NaN,1.0,1.0,66.666667,66.666667,66.666667,75.0,79.591837,1,0,0,2,3,1,66.666667,60.000000,1,0,50.0,66.666667,75.0,79.591837,0,0,0,NaN,50.0,0,0,NaN,<NA>,NaN,NaN,NaN,0,1,66.666667,NaN,0.0,0,NaN,NaN,NaN,<NA>,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,1,1,5,1,1,5,5,1,25,5,0,1,0,6,0,0,0.833333,0,1,1,6,5,0.833333,4,5,80.000000,3,4,75.000000,1,1,66.666667,4,1.0,1.0,0.0,1.0,NaN,NaN,1.0,1.0,66.666667,75.000000,75.000000,87.5,85.422741,2,0,0,3,4,1,75.000000,66.666667,2,0,100.0,66.666667,87.5,85.422741,0,0,0,NaN,50.0,0,0,NaN,<NA>,NaN,NaN,NaN,0,1,66.666667,NaN,0.0,0,NaN,NaN,NaN,

In [4]:
# Dataset Dimensions
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Participants: {df['id'].nunique():,}")
print(f"Study Days: {df['day'].nunique():,}")

# Data Types and Non-Null Counts
display(
    df.dtypes.rename("dtype")
    .to_frame()
    .join(df.count().rename("non_null"))
    .assign(
        missing=lambda x: len(df) - x["non_null"],
        missing_pct=lambda x: (
            (len(df) - x["non_null"]) / len(df) * 100
        ).round(2),
        unique=lambda x: df.nunique()
    )
    .sort_values("missing_pct", ascending=False)
)

# Target Distribution
if "responded" in df.columns:
    print("\nTarget Distribution:")
    display(
        df["responded"]
        .value_counts(dropna=False)
        .rename("count")
        .to_frame()
        .assign(
            percentage=lambda x: (
                x["count"] / len(df) * 100
            ).round(2)
        )
    )

# Participant-Level Distribution
print("\nRows per Participant:")
display(
    df.groupby("id", dropna=False)
    .size()
    .describe()
    .to_frame("rows_per_participant")
)

# Sample Records
display(df.head())

Rows: 30,000
Columns: 92
Participants: 125
Study Days: 40


,dtype,non_null,missing,missing_pct,unique
beep_gap_from_previous_same_day,Int64,25000,5000,16.67,1
prior_same_day_3_prompt_response_rate,float64,25000,5000,16.67,5
prior_same_day_ewm_3_response_rate,float64,25000,5000,16.67,17
prev_response_same_day,float64,25000,5000,16.67,2
prior_same_day_response_rate,float64,25000,5000,16.67,11
prior_same_day_ewm_6_response_rate,float64,25000,5000,16.67,32
prior_same_day_2_prompt_response_rate,float64,25000,5000,16.67,3
prior_3_day_response_rate,float64,29250,750,2.50,61
prior_same_beep_response_rate,float64,29250,750,2.50,407
prior_same_beep_ewm_6_response_rate,float64,29250,750,2.50,17075



Target Distribution:


,count,percentage
responded,,
1,21570,71.9
0,8430,28.1



Rows per Participant:


,rows_per_participant
count,125.0
mean,240.0
std,0.0
min,240.0
25%,240.0
50%,240.0
75%,240.0
max,240.0


,id,day,beep,responded,day_index,beep_index,prompt_index,day_squared,beep_squared,day_beep_interaction,days_since_first_observed,first_observed_day,last_observed_day,prompts_in_day,first_beep_of_day,last_beep_of_day,beep_position,days_since_previous_prompt,beep_difference_from_previous,beep_gap_from_previous_same_day,scheduled_beeps_in_day,scheduled_beep_position,scheduled_beep_position_rate,cumulative_responses,cumulative_prompts,cumulative_response_rate,prior_cumulative_responses,prior_cumulative_prompts,prior_cumulative_response_rate,prior_cumulative_nonresponses,prior_missed_beep_count,prior_smoothed_response_rate,prior_response_history_count,prev_1_response,prev_2_response,prev_3_response,prev_4_response,prev_5_response,prev_6_response,prev_response,prev_response_same_day,prior_3_prompt_response_rate,prior_6_prompt_response_rate,prior_12_prompt_response_rate,prior_ewm_3_prompt_response_rate,prior_ewm_6_prompt_response_rate,prior_response_streak,prior_nonresponse_streak,days_since_previous_response,prior_same_day_responses,prior_same_day_prompts,prior_same_day_nonresponses,prior_same_day_response_rate,prior_same_day_smoothed_response_rate,prior_same_day_response_streak,prior_same_day_nonresponse_streak,prior_same_day_2_prompt_response_rate,prior_same_day_3_prompt_response_rate,prior_same_day_ewm_3_response_rate,prior_same_day_ewm_6_response_rate,prior_same_beep_responses,prior_same_beep_observations,prior_same_beep_nonresponses,prior_same_beep_response_rate,prior_same_beep_smoothed_rate,prior_same_beep_response_streak,prior_same_beep_nonresponse_streak,prev_same_beep_response,days_since_previous_same_beep,prior_same_beep_ewm_3_response_rate,prior_same_beep_ewm_6_response_rate,prev_day_same_beep_response,prev_day_same_beep_available,responded_any_today,daily_response_rate,prior_day_response_rate,prior_days_with_response,prior_days_observed,prior_day_participation_rate,prev_day_responded,prev_day_response_rate,days_since_previous_observed_day,prior_consecutive_active_days,prior_consecutive_inactive_days,prior_response_rate_change,prior_3_day_response_rate,prior_3_day_participation_rate,prior_7_day_response_rate,prior_7_day_participation_rate,prior_ewm_3_day_response_rate,prior_ewm_6_day_response_rate,missed_day
0,1,1,1,1,1,1,1,1,1,1,0,1,0,6,1,0,0.166667,<NA>,<NA>,<NA>,6,1,0.166667,1,1,100.000000,0,0,NaN,0,0,50.000000,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,<NA>,0,0,0,NaN,50.000000,0,0,NaN,NaN,NaN,NaN,0,0,0,NaN,50.0,0,0,NaN,<NA>,NaN,NaN,NaN,0,1,66.666667,NaN,0.0,0,NaN,NaN,NaN,<NA>,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1,1,2,0,1,2,2,1,4,2,0,1,0,6,0,0,0.333333,0,1,1,6,2,0.333333,1,2,50.000000,1,1,100.000000,0,0,66.666667,1,1.0,NaN,NaN,NaN,NaN,NaN,1.0,1.0,100.000000,100.000000,100.000000,100.0,100.000000,1,0,0,1,1,0,100.000000,66.666667,1,0,100.0,100.000000,100.0,100.000000,0,0,0,NaN,50.0,0,0,NaN,<NA>,NaN,NaN,NaN,0,1,66.666667,NaN,0.0,0,NaN,NaN,NaN,<NA>,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,1,1,3,1,1,3,3,1,9,3,0,1,0,6,0,0,0.5,0,1,1,6,3,0.5,2,3,66.666667,1,2,50.000000,1,1,50.000000,2,0.0,1.0,NaN,NaN,NaN,NaN,0.0,0.0,50.000000,50.000000,50.000000,50.0,71.428571,0,1,0,1,2,1,50.000000,50.000000,0,1,50.0,50.000000,50.0,71.428571,0,0,0,NaN,50.0,0,0,NaN,<NA>,NaN,NaN,NaN,0,1,66.666667,NaN,0.0,0,NaN,NaN,NaN,<NA>,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,1,1,4,1,1,4,4,1,16,4,0,1,0,6,0,0,0.666667,0,1,1,6,4,0.666667,3,4,75.000000,2,3,66.666667,1,1,60.000000,3,1.0,0.0,1.0,NaN,NaN,NaN,1.0,1.0,66.666667,66.666667,66.666667,75.0,79.591837,1,0,0,2,3,1,66.666667,60.000000,1,0,50.0,66.666667,75.0,79.591837,0,0,0,NaN,50.0,0,0,NaN,<NA>,NaN,NaN,NaN,0,1,66.666667,NaN,0.0,0,NaN,NaN,NaN,<NA>,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,1,1,5,1,1,5,5,1,25,5,0,1,0,6,0,0,0.833333,0,1,1,6,5,0.833333,4,5,80.000000,3,4,75.000000,1,1,66.666667,4,1.0,1.0,0.0,1.0,NaN,NaN,1.0,1.0,66.666667,75.000000,75.000000,87.5,85.422741,2,0,0,3,4,1,75.000000,66.666667,2,0,100.0,66.666667,87.5,85.422741,0,0,0,NaN,50.0,0,0,NaN,<NA>,NaN,NaN,NaN,0,1,66.666667,NaN,0.0,0,NaN,NaN,NaN,

In [5]:
# Feature Audit
feature_audit = pd.DataFrame({
    "dtype": df.dtypes,
    "unique": df.nunique(dropna=True),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
})

feature_audit["unique_pct"] = (
    feature_audit["unique"] / len(df) * 100
)

feature_audit["constant"] = feature_audit["unique"] <= 1

feature_audit = feature_audit.sort_values(
    ["constant", "missing_pct", "unique"],
    ascending=[False, False, True]
)

display(feature_audit)

,dtype,unique,missing,missing_pct,unique_pct,constant
beep_gap_from_previous_same_day,Int64,1,5000,16.67,0.003333,True
days_since_previous_same_beep,Int64,1,750,2.50,0.003333,True
days_since_previous_observed_day,Int64,1,750,2.50,0.003333,True
prompts_in_day,Int64,1,0,0.00,0.003333,True
scheduled_beeps_in_day,Int64,1,0,0.00,0.003333,True
prev_response_same_day,float64,2,5000,16.67,0.006667,False
prior_same_day_2_prompt_response_rate,float64,3,5000,16.67,0.010000,False
prior_same_day_3_prompt_response_rate,float64,5,5000,16.67,0.016667,False
prior_same_day_response_rate,float64,11,5000,16.67,0.036667,False
prior_same_day_ewm_3_response_rate,float64,17,5000,16.67,0.056667,False


In [6]:
model_df = df.copy()

# Split boundaries
TRAIN_LAST_DAY = 28
VALIDATION_LAST_DAY = 34
TEST_FIRST_DAY = 35

# Create target: 1 = nonresponse, 0 = response
model_df["nonresponse"] = (
    1 - model_df["responded"]
).astype(int)

# Standardize day using training-period statistics
train_mask = model_df["day"] <= TRAIN_LAST_DAY

day_mean = model_df.loc[train_mask, "day"].mean()
day_std = model_df.loc[train_mask, "day"].std()

day_squared_mean = model_df.loc[
    train_mask, "day_squared"
].mean()

day_squared_std = model_df.loc[
    train_mask, "day_squared"
].std()

model_df["day_z"] = (
    model_df["day"] - day_mean
) / day_std

model_df["day_squared_z"] = (
    model_df["day_squared"] - day_squared_mean
) / day_squared_std

print("Modeling DataFrame:", model_df.shape)
print("Nonresponse prevalence:", model_df["nonresponse"].mean())

Modeling DataFrame: (30000, 95)
Nonresponse prevalence: 0.281


In [7]:
ID_COL = "id"
DAY_COL = "day"
BEEP_COL = "beep"
TARGET_COL = "nonresponse"

TRAIN_LAST_DAY = 28
VALIDATION_LAST_DAY = 34
TEST_FIRST_DAY = 35

# Transition inputs. Beep is represented categorically by dummy variables.
TRANSITION_FEATURES_BASE = [
    "day_z",
    "day_squared_z",
    "is_day_boundary",
]
BEEP_DUMMY_FEATURES = [f"beep_{value}" for value in range(2, 7)]
TRANSITION_FEATURES = TRANSITION_FEATURES_BASE + BEEP_DUMMY_FEATURES

# State-specific Bernoulli emission predictors.
# Edit this list only if your actual engineered column names differ.
HISTORY_EMISSION_FEATURES = [
    "prev_1_response",
    "prev_6_response",
    "prior_3_day_response_rate",
    "prior_cumulative_response_rate",
    "days_since_previous_response",
    "prior_same_day_nonresponse_streak",
]

# Two alternatives:
# - intercept_only: each state has its own baseline nonresponse probability.
# - history_logistic: each state has its own logistic regression on history.
EMISSION_SPECS = {
    "intercept_only": [],
    "history_logistic": HISTORY_EMISSION_FEATURES,
}

STATE_COUNTS = [2, 3]
N_RESTARTS = 3
MAX_EM_ITER = 100
EM_TOL = 1e-4
RANDOM_SEEDS = [17, 41, 73, 101, 137, 173]

OUTPUT_DIR = Path("outputs") / "iohmm_nonresponse"

# Optional: CSV containing row-level Elastic Net probabilities from your
# existing benchmark. It must contain id, day, beep, and a probability column.
# Leave as None if you do not have the saved row-level predictions yet.
ELASTIC_NET_PREDICTIONS_PATH = None
ELASTIC_NET_PROBABILITY_COL = "elastic_net_probability"


In [8]:
def prepare_model_frame(model_df: pd.DataFrame) -> pd.DataFrame:
    """Validate the engineered frame and add model-only schedule indicators."""

    required = [
        ID_COL,
        DAY_COL,
        BEEP_COL,
        TARGET_COL,
        "day_z",
        "day_squared_z",
        *HISTORY_EMISSION_FEATURES,
    ]

    missing = [column for column in required if column not in model_df.columns]
    if missing:
        raise KeyError(
            "model_df is missing required columns:\n"
            + "\n".join(f"  - {column}" for column in missing)
            + "\n\nCheck the feature names in the configuration section."
        )

    df = model_df.copy()

    df[DAY_COL] = pd.to_numeric(df[DAY_COL], errors="raise")
    df[BEEP_COL] = pd.to_numeric(df[BEEP_COL], errors="raise")
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="raise")

    if df[[ID_COL, DAY_COL, BEEP_COL, TARGET_COL]].isna().any().any():
        raise ValueError("id/day/beep/nonresponse contains missing values.")

    observed_targets = set(df[TARGET_COL].unique())
    if not observed_targets.issubset({0, 1}):
        raise ValueError(
            f"{TARGET_COL} must contain only 0 and 1; found {observed_targets}"
        )

    df[TARGET_COL] = df[TARGET_COL].astype(int)

    # Ensure one row per scheduled participant-day-beep prompt.
    duplicate_mask = df.duplicated([ID_COL, DAY_COL, BEEP_COL], keep=False)
    if duplicate_mask.any():
        examples = df.loc[
            duplicate_mask, [ID_COL, DAY_COL, BEEP_COL]
        ].head(10)
        raise ValueError(
            "Duplicate participant/day/beep rows found. Resolve these before "
            "modeling. Examples:\n"
            f"{examples.to_string(index=False)}"
        )

    df = df.sort_values(
        [ID_COL, DAY_COL, BEEP_COL],
        kind="mergesort",
    ).reset_index(drop=True)

    # A transition from the previous row into this row crosses a day boundary.
    # IOHMM uses transition covariates from the destination/current row.
    previous_day = df.groupby(ID_COL, sort=False)[DAY_COL].shift(1)
    df["is_day_boundary"] = (
        previous_day.notna() & df[DAY_COL].ne(previous_day)
    ).astype(int)

    # Categorical beep encoding: beep 1 is the reference category.
    for beep_value in range(2, 7):
        df[f"beep_{beep_value}"] = (
            df[BEEP_COL].eq(beep_value)
        ).astype(int)

    # Validate schedule values rather than silently creating malformed dummies.
    unexpected_beeps = sorted(set(df[BEEP_COL].unique()) - set(range(1, 7)))
    if unexpected_beeps:
        raise ValueError(
            "This script is configured for beep values 1-6. "
            f"Unexpected values: {unexpected_beeps}"
        )

    return df


def chronological_split(df: pd.DataFrame):
    """Return the requested fixed day-based train/validation/test split."""

    train = df.loc[df[DAY_COL] <= TRAIN_LAST_DAY].copy()
    validation = df.loc[
        df[DAY_COL].between(TRAIN_LAST_DAY + 1, VALIDATION_LAST_DAY)
    ].copy()
    test = df.loc[df[DAY_COL] >= TEST_FIRST_DAY].copy()

    if train.empty or validation.empty or test.empty:
        raise ValueError(
            "One of train/validation/test is empty. "
            "Check the day range and split boundaries."
        )

    train_ids = set(train[ID_COL].unique())
    validation_ids = set(validation[ID_COL].unique())
    test_ids = set(test[ID_COL].unique())

    if not validation_ids.issubset(train_ids):
        raise ValueError(
            "Some validation participants have no training history: "
            f"{sorted(validation_ids - train_ids)[:20]}"
        )
    if not test_ids.issubset(train_ids | validation_ids):
        raise ValueError(
            "Some test participants have no training/validation history: "
            f"{sorted(test_ids - (train_ids | validation_ids))[:20]}"
        )

    print("\nSPLIT SUMMARY")
    print("-" * 65)
    for name, part in [
        ("Train", train),
        ("Validation", validation),
        ("Test", test),
    ]:
        print(
            f"{name:12s} rows={len(part):7,d} | "
            f"participants={part[ID_COL].nunique():4,d} | "
            f"days={part[DAY_COL].min():g}-{part[DAY_COL].max():g} | "
            f"prevalence={part[TARGET_COL].mean():.4f}"
        )

    return train, validation, test

In [9]:
def impute_from_reference(
    reference_df: pd.DataFrame,
    target_df: pd.DataFrame,
    columns: list[str],
):
    """
    Impute numeric model inputs using medians estimated on reference_df only.
    Returns a copied target frame and the medians used.
    """

    output = target_df.copy()
    medians = {}

    for column in columns:
        reference_values = pd.to_numeric(
            reference_df[column], errors="coerce"
        )
        median = reference_values.median()

        if pd.isna(median):
            raise ValueError(
                f"Feature {column!r} is entirely missing in the reference data."
            )

        target_values = pd.to_numeric(output[column], errors="coerce")
        output[column] = target_values.fillna(median).astype(float)
        medians[column] = float(median)

    if output[columns].isna().any().any():
        raise ValueError("NaNs remain in model predictors after imputation.")

    return output, medians

In [10]:
def make_sequences(
    df: pd.DataFrame,
    model_columns: list[str],
) -> list[pd.DataFrame]:
    """One chronologically ordered DataFrame per participant."""

    sequences = []

    for _, participant_df in df.groupby(ID_COL, sort=False):
        sequence = (
            participant_df
            .sort_values([DAY_COL, BEEP_COL], kind="mergesort")
            [model_columns]
            .reset_index(drop=True)
            .copy()
        )

        if len(sequence) == 0:
            continue

        sequences.append(sequence)

    return sequences


def make_sequence_lookup(
    df: pd.DataFrame,
    model_columns: list[str],
) -> dict:
    """Participant ID -> ordered sequence DataFrame."""

    output = {}

    for participant_id, participant_df in df.groupby(ID_COL, sort=False):
        output[participant_id] = (
            participant_df
            .sort_values([DAY_COL, BEEP_COL], kind="mergesort")
            [model_columns]
            .reset_index(drop=True)
            .copy()
        )

    return output

In [11]:
def build_iohmm(
    num_states: int,
    emission_features: list[str],
) -> UnSupervisedIOHMM:
    """Construct the selected Mogeng Yin IOHMM architecture."""

    model = UnSupervisedIOHMM(
        num_states=num_states,
        EM_tol=EM_TOL,
        max_EM_iter=MAX_EM_ITER,
    )

    # The package uses CrossEntropyMNL for initial and transition models.
    # Both are configured identically, including the number of classes.
    initial_model = CrossEntropyMNL(
        solver="lbfgs",
        n_classes=num_states,
        max_iter=300,
    )
    transition_model = CrossEntropyMNL(
        solver="lbfgs",
        n_classes=num_states,
        max_iter=300,
    )

    emission_model = GLM(
        family=Binomial(),
        solver="IRLS",
        max_iter=200,
        tol=1e-8,
        est_stderr=False,
    )

    model.set_models(
        model_initial=initial_model,
        model_transition=transition_model,
        model_emissions=[emission_model],
    )

    # Empty initial covariates means an intercept-only initial-state model.
    model.set_inputs(
        covariates_initial=[],
        covariates_transition=TRANSITION_FEATURES,
        covariates_emissions=[emission_features],
    )

    model.set_outputs(
        responses_emissions=[[TARGET_COL]]
    )

    return model


In [12]:
def fit_one_restart(
    train_df: pd.DataFrame,
    model_columns: list[str],
    num_states: int,
    emission_features: list[str],
    seed: int,
):
    """Fit one EM initialization using the package's own training routine."""

    np.random.seed(seed)

    model = build_iohmm(
        num_states=num_states,
        emission_features=emission_features,
    )

    sequences = make_sequences(train_df, model_columns)

    model.set_data(sequences)
    model.train()

    if not getattr(model, "trained", False):
        raise RuntimeError("IOHMM did not mark itself as trained.")

    if not np.isfinite(model.log_likelihood):
        raise RuntimeError(
            f"Non-finite training likelihood: {model.log_likelihood}"
        )

    return model

In [13]:

def emission_probability_by_state(
    model,
    row: pd.DataFrame,
    emission_features: list[str],
) -> np.ndarray:
    """Return P(nonresponse=1 | state=k, current pre-outcome inputs)."""

    X = row[emission_features].to_numpy(dtype=float)

    probabilities = []
    for state in range(model.num_states):
        probability = np.asarray(
            model.model_emissions[state][0].predict(X),
            dtype=float,
        ).reshape(-1)[0]
        probabilities.append(np.clip(probability, 1e-8, 1.0 - 1e-8))

    return np.asarray(probabilities)

In [14]:
def transition_matrix_for_row(
    model,
    row: pd.DataFrame,
) -> np.ndarray:
    """
    Return A_t where A_t[j, k] = P(S_t=k | S_(t-1)=j, X_t).
    Transition covariates belong to the destination/current prompt.
    """

    X = row[TRANSITION_FEATURES].to_numpy(dtype=float)
    matrix = np.vstack([
        np.exp(model.model_transition[previous_state].predict_log_proba(X)[0])
        for previous_state in range(model.num_states)
    ])

    matrix = np.clip(matrix, 1e-12, 1.0)
    matrix = matrix / matrix.sum(axis=1, keepdims=True)

    return matrix

In [15]:

def filtered_training_terminal_state(
    model,
    training_sequence: pd.DataFrame,
    emission_features: list[str],
) -> np.ndarray:
    """
    Use the package's forward recursion to obtain the filtered state
    distribution after the final observed training prompt.

    This is filtering, not smoothing: no backward probabilities are used.
    """

    if training_sequence.empty:
        raise ValueError("Cannot initialize from an empty training sequence.")

    y = training_sequence[TARGET_COL].to_numpy(dtype=int)

    # Initial distribution at the first training observation.
    X_initial = np.empty((1, 0), dtype=float)
    log_initial = model.model_initial.predict_log_proba(X_initial)[0]

    # Transition probabilities for rows 2...T.
    if len(training_sequence) > 1:
        X_transition = training_sequence.iloc[1:][
            TRANSITION_FEATURES
        ].to_numpy(dtype=float)

        log_transition = np.stack(
            [
                model.model_transition[previous_state].predict_log_proba(
                    X_transition
                )
                for previous_state in range(model.num_states)
            ],
            axis=1,
        )
        # Shape: (T-1, previous_state, next_state)
    else:
        log_transition = np.empty(
            (0, model.num_states, model.num_states),
            dtype=float,
        )

    # State-specific emission log likelihoods for observed training outcomes.
    X_emission = training_sequence[emission_features].to_numpy(dtype=float)
    log_emission = np.column_stack([
        model.model_emissions[state][0].loglike_per_sample(
            X_emission,
            y,
        )
        for state in range(model.num_states)
    ])

    # Package's forward() returns log alpha_t(k), without backward smoothing.
    log_alpha = forward(
        log_initial,
        log_transition,
        log_emission,
        {},
    )

    terminal_log_alpha = log_alpha[-1] - logsumexp(log_alpha[-1])
    terminal_alpha = np.exp(terminal_log_alpha)

    if not np.isfinite(terminal_alpha).all():
        raise RuntimeError("Non-finite terminal filtered state distribution.")

    return terminal_alpha / terminal_alpha.sum()

In [16]:
def sequential_filtered_predictions(
    model,
    history_df: pd.DataFrame,
    evaluation_df: pd.DataFrame,
    model_columns: list[str],
    emission_features: list[str],
) -> pd.DataFrame:
    """
    Sequential one-step-ahead predictions for evaluation_df.

    At each prompt:
      1. Propagate the previous filtered state distribution through A_t.
      2. Predict P(nonresponse_t=1) before seeing nonresponse_t.
      3. Observe the actual target.
      4. Update the filtered state distribution.
      5. Continue to the next prompt.

    Each participant has an independent state distribution.
    """

    history_lookup = make_sequence_lookup(history_df, model_columns)
    evaluation_lookup = make_sequence_lookup(evaluation_df, model_columns)

    records = []

    for participant_id, eval_sequence in evaluation_lookup.items():
        if participant_id not in history_lookup:
            raise ValueError(
                f"Participant {participant_id!r} has no prior history."
            )

        history_sequence = history_lookup[participant_id]

        alpha = filtered_training_terminal_state(
            model,
            history_sequence,
            emission_features,
        )

        for row_index in range(len(eval_sequence)):
            row = eval_sequence.iloc[[row_index]]

            transition_matrix = transition_matrix_for_row(model, row)

            # State distribution before observing the current prompt outcome.
            state_prior = alpha @ transition_matrix
            state_prior = np.clip(state_prior, 1e-12, 1.0)
            state_prior = state_prior / state_prior.sum()

            # State-conditional Bernoulli probabilities.
            state_p_nonresponse = emission_probability_by_state(
                model,
                row,
                emission_features,
            )

            # The actual one-step-ahead prediction.
            predicted_probability = float(
                np.dot(state_prior, state_p_nonresponse)
            )
            predicted_probability = float(
                np.clip(predicted_probability, 1e-8, 1.0 - 1e-8)
            )

            actual_y = int(row[TARGET_COL].iloc[0])

            # Only now use the observed outcome to update the state posterior.
            if actual_y == 1:
                likelihood = state_p_nonresponse
            else:
                likelihood = 1.0 - state_p_nonresponse

            unnormalized_alpha = state_prior * likelihood
            normalizer = unnormalized_alpha.sum()

            if not np.isfinite(normalizer) or normalizer <= 0:
                raise RuntimeError(
                    "Invalid filtering update; state likelihood collapsed."
                )

            alpha = unnormalized_alpha / normalizer

            record = {
                ID_COL: participant_id,
                DAY_COL: row[DAY_COL].iloc[0],
                BEEP_COL: row[BEEP_COL].iloc[0],
                "y_true": actual_y,
                "predicted_probability": predicted_probability,
                "state_emission_probability_mean": float(
                    np.mean(state_p_nonresponse)
                ),
            }

            for state in range(model.num_states):
                record[f"state_prior_{state}"] = float(state_prior[state])
                record[f"state_filtered_{state}"] = float(alpha[state])
                record[f"state_p_nonresponse_{state}"] = float(
                    state_p_nonresponse[state]
                )

            records.append(record)

    predictions = pd.DataFrame(records)
    predictions = predictions.sort_values(
        [ID_COL, DAY_COL, BEEP_COL]
    ).reset_index(drop=True)

    return predictions

In [17]:
def calculate_metrics(
    y_true,
    probabilities,
) -> dict:
    """Probability-focused evaluation metrics."""

    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1.0 - 1e-8,
    )

    metrics = {
        "n": int(len(y_true)),
        "prevalence": float(np.mean(y_true)),
        "average_precision": float(
            average_precision_score(y_true, probabilities)
        ),
        "brier_score": float(
            brier_score_loss(y_true, probabilities)
        ),
        "log_loss": float(
            log_loss(y_true, probabilities, labels=[0, 1])
        ),
    }

    metrics["roc_auc"] = (
        float(roc_auc_score(y_true, probabilities))
        if len(np.unique(y_true)) == 2
        else None
    )

    return metrics


def calibration_table(
    predictions: pd.DataFrame,
    n_bins: int = 10,
) -> pd.DataFrame:
    """Quantile-binned calibration table."""

    y = predictions["y_true"].to_numpy(dtype=int)
    p = predictions["predicted_probability"].to_numpy(dtype=float)

    observed, predicted = calibration_curve(
        y,
        p,
        n_bins=n_bins,
        strategy="quantile",
    )

    # Also provide bin counts using qcut, useful when bins have unequal sizes.
    bin_ids = pd.qcut(
        p,
        q=n_bins,
        labels=False,
        duplicates="drop",
    )

    grouped = (
        pd.DataFrame({
            "bin": bin_ids,
            "y_true": y,
            "predicted_probability": p,
        })
        .groupby("bin", observed=True)
        .agg(
            n=("y_true", "size"),
            mean_predicted=("predicted_probability", "mean"),
            observed_rate=("y_true", "mean"),
        )
        .reset_index()
    )

    return grouped

In [18]:
def state_occupancy_table(
    predictions: pd.DataFrame,
    num_states: int,
) -> pd.DataFrame:
    """Occupancy and state-conditional response diagnostics."""

    rows = []
    filtered_columns = [f"state_filtered_{s}" for s in range(num_states)]
    prior_columns = [f"state_prior_{s}" for s in range(num_states)]

    for state in range(num_states):
        filtered = predictions[filtered_columns[state]].to_numpy()
        prior = predictions[prior_columns[state]].to_numpy()
        emission_p = predictions[
            f"state_p_nonresponse_{state}"
        ].to_numpy()

        rows.append({
            "state": state,
            "mean_filtered_occupancy": float(filtered.mean()),
            "mean_prior_occupancy": float(prior.mean()),
            "argmax_occupancy": float(
                (
                    predictions[filtered_columns].to_numpy().argmax(axis=1)
                    == state
                ).mean()
            ),
            "mean_state_nonresponse_probability": float(
                np.average(emission_p, weights=filtered)
            ),
            "unweighted_mean_state_nonresponse_probability": float(
                emission_p.mean()
            ),
        })

    return pd.DataFrame(rows)


In [19]:
def transition_diagnostics(
    model,
    evaluation_df: pd.DataFrame,
) -> pd.DataFrame:
    """Average fitted transition probabilities overall and by day boundary."""

    rows = []

    subsets = {
        "all": evaluation_df,
        "within_day": evaluation_df.loc[
            evaluation_df["is_day_boundary"].eq(0)
        ],
        "day_boundary": evaluation_df.loc[
            evaluation_df["is_day_boundary"].eq(1)
        ],
    }

    for subset_name, subset in subsets.items():
        if subset.empty:
            continue

        for _, row in subset.iterrows():
            transition_matrix = transition_matrix_for_row(
                model,
                row.to_frame().T,
            )

            for from_state in range(model.num_states):
                for to_state in range(model.num_states):
                    rows.append({
                        "subset": subset_name,
                        "from_state": from_state,
                        "to_state": to_state,
                        "probability": transition_matrix[
                            from_state, to_state
                        ],
                    })

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .groupby(
            ["subset", "from_state", "to_state"],
            as_index=False,
        )["probability"]
        .mean()
    )

In [20]:
def coefficient_summary(model) -> dict:
    """Export raw model coefficients without assuming their internal layout."""

    output = {
        "initial_model_coefficients": np.asarray(
            model.model_initial.coef
        ).tolist(),
        "transition_model_coefficients": [],
        "emission_model_coefficients": [],
    }

    for state, transition_model in enumerate(model.model_transition):
        output["transition_model_coefficients"].append({
            "previous_state": state,
            "coefficients": np.asarray(
                transition_model.coef
            ).tolist(),
        })

    for state in range(model.num_states):
        emission = model.model_emissions[state][0]
        output["emission_model_coefficients"].append({
            "state": state,
            "coefficients": np.asarray(emission.coef).tolist(),
            "standard_errors": (
                None if emission.stderr is None
                else np.asarray(emission.stderr).tolist()
            ),
        })

    return output


In [ ]:
def run_iohmm_experiment(
    model_df: pd.DataFrame,
    output_dir: Path = OUTPUT_DIR,
):
    """Run validation model comparison and final chronological test."""

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    df = prepare_model_frame(model_df)
    train_raw, validation_raw, test_raw = chronological_split(df)

    # Model feature columns: ID is kept outside the IOHMM sequence DataFrames.
    model_feature_columns = list(dict.fromkeys(
        TRANSITION_FEATURES + HISTORY_EMISSION_FEATURES
    ))

    # -----------------------------
    # VALIDATION MODEL COMPARISON
    # -----------------------------

    validation_results = []
    fitted_validation_models = {}

    configurations = [
        (num_states, emission_name, emission_features)
        for num_states in STATE_COUNTS
        for emission_name, emission_features in EMISSION_SPECS.items()
    ]

    total_fits = len(configurations) * N_RESTARTS

    with tqdm(
        total=total_fits,
        desc="IOHMM validation fits",
        unit="fit",
    ) as pbar:

        for num_states, emission_name, emission_features in configurations:

            print(
                f"\nVALIDATION: states={num_states}, "
                f"emission={emission_name}"
            )

            # Train-only imputation for this validation experiment.
            train, _ = impute_from_reference(
                train_raw,
                train_raw,
                model_feature_columns,
            )
            validation, _ = impute_from_reference(
                train_raw,
                validation_raw,
                model_feature_columns,
            )

            model_columns = list(dict.fromkeys(
                [TARGET_COL] + model_feature_columns
            ))

            successful_restarts = 0

            for restart in range(N_RESTARTS):
                seed = RANDOM_SEEDS[restart % len(RANDOM_SEEDS)]

                pbar.set_postfix(
                    states=num_states,
                    emission=emission_name,
                    restart=f"{restart + 1}/{N_RESTARTS}",
                )

                try:
                    model = fit_one_restart(
                        train_df=train,
                        model_columns=model_columns,
                        num_states=num_states,
                        emission_features=emission_features,
                        seed=seed,
                    )

                    validation_predictions = sequential_filtered_predictions(
                        model=model,
                        history_df=train,
                        evaluation_df=validation,
                        model_columns=model_columns,
                        emission_features=emission_features,
                    )

                    metrics = calculate_metrics(
                        validation_predictions["y_true"],
                        validation_predictions["predicted_probability"],
                    )

                    result = {
                        "num_states": num_states,
                        "emission_spec": emission_name,
                        "restart": restart,
                        "seed": seed,
                        "training_log_likelihood": float(
                            model.log_likelihood
                        ),
                        "status": "success",
                        **metrics,
                    }

                    validation_results.append(result)

                    fitted_validation_models[
                        (num_states, emission_name, restart)
                    ] = {
                        "model": model,
                        "predictions": validation_predictions,
                        "metrics": metrics,
                        "seed": seed,
                        "model_columns": model_columns,
                    }

                    successful_restarts += 1

                    print(
                        f"  restart={restart} seed={seed} "
                        f"AP={metrics['average_precision']:.4f} "
                        f"Brier={metrics['brier_score']:.4f} "
                        f"LogLoss={metrics['log_loss']:.4f} "
                        f"ROC-AUC={metrics['roc_auc']}"
                    )

                except Exception as error:
                    logging.exception(
                        "Failed: states=%s emission=%s restart=%s",
                        num_states,
                        emission_name,
                        restart,
                    )

                    validation_results.append({
                        "num_states": num_states,
                        "emission_spec": emission_name,
                        "restart": restart,
                        "seed": seed,
                        "status": "failed",
                        "error": repr(error),
                    })

                finally:
                    pbar.update(1)

            if successful_restarts == 0:
                print("  No successful restarts for this configuration.")

    results_df = pd.DataFrame(validation_results)

    results_df.to_csv(
        output_dir / "validation_model_comparison_all_restarts.csv",
        index=False,
    )

    successful = results_df.loc[
        results_df["status"].eq("success")
    ].copy()

    if successful.empty:
        raise RuntimeError(
            "All IOHMM fits failed. Inspect the error rows in "
            "validation_model_comparison_all_restarts.csv."
        )

    # AP is the primary selection metric; Brier and log loss break close ties.
    successful = successful.sort_values(
        [
            "average_precision",
            "brier_score",
            "log_loss",
            "roc_auc",
        ],
        ascending=[False, True, True, False],
        kind="mergesort",
    ).reset_index(drop=True)

    successful.to_csv(
        output_dir / "validation_model_comparison_ranked.csv",
        index=False,
    )

    winner = successful.iloc[0]

    winner_key = (
        int(winner["num_states"]),
        str(winner["emission_spec"]),
        int(winner["restart"]),
    )

    winner_fit = fitted_validation_models[winner_key]

    best_validation_predictions = winner_fit["predictions"]

    best_validation_predictions.to_csv(
        output_dir / "best_validation_predictions.csv",
        index=False,
    )

    validation_metrics = winner_fit["metrics"]

    with open(output_dir / "best_validation_metrics.json", "w") as file:
        json.dump(validation_metrics, file, indent=2)

    best_num_states = int(winner["num_states"])
    best_emission_name = str(winner["emission_spec"])
    best_emission_features = EMISSION_SPECS[best_emission_name]
    selected_seed = int(winner["seed"])
    model_columns = winner_fit["model_columns"]

    print("\nSELECTED VALIDATION CONFIGURATION")
    print("-" * 65)
    print(
        f"States: {best_num_states}\n"
        f"Emission: {best_emission_name}\n"
        f"Seed: {selected_seed}\n"
        f"Validation AP: {validation_metrics['average_precision']:.4f}\n"
        f"Validation Brier: {validation_metrics['brier_score']:.4f}\n"
        f"Validation Log Loss: {validation_metrics['log_loss']:.4f}\n"
        f"Validation ROC-AUC: {validation_metrics['roc_auc']}"
    )

    # -----------------------------
    # FINAL REFIT: DAYS 1-34
    # -----------------------------

    train_validation_raw = df.loc[
        df[DAY_COL] <= VALIDATION_LAST_DAY
    ].copy()

    # Re-estimate imputation medians using days 1-34 only.
    train_validation, final_medians = impute_from_reference(
        train_validation_raw,
        train_validation_raw,
        model_feature_columns,
    )

    test, _ = impute_from_reference(
        train_validation_raw,
        test_raw,
        model_feature_columns,
    )

    np.random.seed(selected_seed)

    with tqdm(
        total=1,
        desc="Final IOHMM refit",
        unit="fit",
    ) as pbar:

        final_model = fit_one_restart(
            train_df=train_validation,
            model_columns=model_columns,
            num_states=best_num_states,
            emission_features=best_emission_features,
            seed=selected_seed,
        )

        pbar.update(1)

    # -----------------------------
    # FINAL SEQUENTIAL TEST
    # -----------------------------

    test_predictions = sequential_filtered_predictions(
        model=final_model,
        history_df=train_validation,
        evaluation_df=test,
        model_columns=model_columns,
        emission_features=best_emission_features,
    )

    test_metrics = calculate_metrics(
        test_predictions["y_true"],
        test_predictions["predicted_probability"],
    )

    test_predictions.to_csv(
        output_dir / "final_test_predictions.csv",
        index=False,
    )

    with open(output_dir / "final_test_metrics.json", "w") as file:
        json.dump(test_metrics, file, indent=2)

    # -----------------------------
    # STATE AND TRANSITION DIAGNOSTICS
    # -----------------------------

    validation_occupancy = state_occupancy_table(
        best_validation_predictions,
        best_num_states,
    )

    validation_occupancy.to_csv(
        output_dir / "validation_state_occupancy.csv",
        index=False,
    )

    test_occupancy = state_occupancy_table(
        test_predictions,
        best_num_states,
    )

    test_occupancy.to_csv(
        output_dir / "test_state_occupancy.csv",
        index=False,
    )

    test_transition_diagnostics = transition_diagnostics(
        final_model,
        test,
    )

    test_transition_diagnostics.to_csv(
        output_dir / "test_transition_diagnostics.csv",
        index=False,
    )

    coefficients = coefficient_summary(final_model)

    with open(output_dir / "final_model_coefficients.json", "w") as file:
        json.dump(coefficients, file, indent=2)

    with open(output_dir / "final_imputation_medians.json", "w") as file:
        json.dump(final_medians, file, indent=2)

    # Calibration and probability distribution plots/data.
    save_prediction_plots(
        best_validation_predictions,
        output_dir,
        "validation",
    )

    save_prediction_plots(
        test_predictions,
        output_dir,
        "test",
    )

    # Optional direct comparison with saved Elastic Net predictions.
    elastic_net_comparison = compare_elastic_net_if_available(
        test_predictions,
        output_dir,
    )

    # Save fitted final model. If serialization fails, keep all other outputs.
    try:
        joblib.dump(
            final_model,
            output_dir / "final_iohmm_model.joblib",
        )
    except Exception:
        logging.exception("Could not serialize final IOHMM with joblib.")

    manifest = {
        "package": "IOHMM (Mogeng Yin)",
        "num_states": best_num_states,
        "emission_spec": best_emission_name,
        "emission_features": best_emission_features,
        "transition_features": TRANSITION_FEATURES,
        "initial_features": [],
        "selected_validation_seed": selected_seed,
        "n_restarts_per_configuration": N_RESTARTS,
        "max_em_iterations": MAX_EM_ITER,
        "split": {
            "train_days": [1, TRAIN_LAST_DAY],
            "validation_days": [
                TRAIN_LAST_DAY + 1,
                VALIDATION_LAST_DAY,
            ],
            "test_days": [
                TEST_FIRST_DAY,
                int(test[DAY_COL].max()),
            ],
        },
        "validation_metrics": validation_metrics,
        "test_metrics": test_metrics,
        "final_training_log_likelihood": float(
            final_model.log_likelihood
        ),
        "n_train_validation_rows": int(len(train_validation)),
        "n_test_rows": int(len(test)),
        "n_participants": int(df[ID_COL].nunique()),
        "elastic_net_comparison_available": (
            elastic_net_comparison is not None
        ),
    }

    with open(output_dir / "run_manifest.json", "w") as file:
        json.dump(manifest, file, indent=2)

    print("\nFINAL TEST RESULTS")
    print("-" * 65)

    for metric, value in test_metrics.items():
        print(f"{metric:24s}: {value}")

    print("\nSTATE OCCUPANCY — TEST")
    print(test_occupancy.to_string(index=False))

    print("\nTRANSITION DIAGNOSTICS — TEST")
    print(test_transition_diagnostics.to_string(index=False))

    print(f"\nAll outputs saved to: {output_dir.resolve()}")

    return {
        "validation_results": results_df,
        "best_validation_predictions": best_validation_predictions,
        "test_predictions": test_predictions,
        "validation_metrics": validation_metrics,
        "test_metrics": test_metrics,
        "final_model": final_model,
        "output_dir": output_dir,
    }

In [ ]:
results = run_iohmm_experiment(model_df)


SPLIT SUMMARY
-----------------------------------------------------------------
Train        rows= 21,000 | participants= 125 | days=1-28 | prevalence=0.2753
Validation   rows=  4,500 | participants= 125 | days=29-34 | prevalence=0.2993
Test         rows=  4,500 | participants= 125 | days=35-40 | prevalence=0.2891

VALIDATION: states=2, emission=intercept_only


c:\Users\adria\anaconda3\envs\openesm-nonresponse\Lib\site-packages\IOHMM\IOHMM.py:240: RuntimeWarning: divide by zero encountered in log
  log_gamma = np.log(np.zeros((df.shape[0], self.num_states)))
c:\Users\adria\anaconda3\envs\openesm-nonresponse\Lib\site-packages\IOHMM\IOHMM.py:267: RuntimeWarning: divide by zero encountered in log
  log_epsilon = np.log(np.zeros((df.shape[0] - 1, self.num_states, self.num_states)))
c:\Users\adria\anaconda3\envs\openesm-nonresponse\Lib\site-packages\IOHMM\linear_models.py:855: RuntimeWarning: divide by zero encountered in scalar divide
  C = np.float64(1) / self.alpha
c:\Users\adria\anaconda3\envs\openesm-nonresponse\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a f

  No successful restarts for this configuration.

VALIDATION: states=2, emission=history_logistic


c:\Users\adria\anaconda3\envs\openesm-nonresponse\Lib\site-packages\IOHMM\IOHMM.py:240: RuntimeWarning: divide by zero encountered in log
  log_gamma = np.log(np.zeros((df.shape[0], self.num_states)))
c:\Users\adria\anaconda3\envs\openesm-nonresponse\Lib\site-packages\IOHMM\IOHMM.py:267: RuntimeWarning: divide by zero encountered in log
  log_epsilon = np.log(np.zeros((df.shape[0] - 1, self.num_states, self.num_states)))
c:\Users\adria\anaconda3\envs\openesm-nonresponse\Lib\site-packages\IOHMM\linear_models.py:855: RuntimeWarning: divide by zero encountered in scalar divide
  C = np.float64(1) / self.alpha
c:\Users\adria\anaconda3\envs\openesm-nonresponse\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a f

In [ ]:
print(results.keys())